# MLP-Mixer Native Adapter v1

目标：把 `mlp_mixer_mse` 从 proxy feature adapter 推进到 `src/model` 的 native sequence adapter。Notebook 只保留实验流程和中文结论；可复用训练、回测、analysis 逻辑都调用 `src/`。外部论文只保存 URL 与自有摘要，不复制全文。

In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "AGENTS.md").exists() and (path / "src").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model.capabilities import model_capability_table, model_fidelity_summary
from src.model.matrix import run_quantbench_matrix_smoke
from src.model.sequence import MLPMixerSequenceRegressor
from src.model.training import fit_estimator_for_model

OUTPUT_DIR = REPO_ROOT / "research" / "mlp_mixer_native_adapter_v1"
CARD_DIR = OUTPUT_DIR / "matrix_card"

outputs = run_quantbench_matrix_smoke(
    card_dir=CARD_DIR,
    start="2026-01-01",
    end="2026-06-09",
    train_end="2026-04-30",
    model_names=("ridge_mse", "patch_tst_mse", "autoformer_mse", "fedformer_mse", "mlp_mixer_mse"),
)

frame = load_soxl_support_frame(start="2026-01-01", end="2026-06-09")
dataset = build_supervised_dataset(frame, horizon=1)
train = dataset.frame.loc[dataset.frame["date"] <= "2026-04-30"].copy()
estimator, constant_prediction = fit_estimator_for_model(
    "mlp_mixer_mse",
    train=train,
    feature_columns=dataset.feature_columns[:5],
)
capabilities = model_capability_table(("ridge_mse", "patch_tst_mse", "autoformer_mse", "fedformer_mse", "mlp_mixer_mse"))
capability = capabilities.loc[capabilities["model"].eq("mlp_mixer_mse")].iloc[0].to_dict()
comparison = outputs.model_comparison.to_dict(orient="records")
mixer_row = next(row for row in comparison if row["model"] == "mlp_mixer_mse")

assert isinstance(estimator, MLPMixerSequenceRegressor)
assert hasattr(estimator, "model_")
assert estimator.lookback_ == 5
assert estimator.feature_count_ == 5
assert capability["paper_fidelity"] == "native_sequence"
assert capability["is_proxy"] is False

summary = {
    "schema_version": "research.mlp_mixer_native_adapter_v1",
    "status": "completed",
    "paper_reference": {
        "source_id": "arxiv:2105.01601",
        "url": "https://arxiv.org/abs/2105.01601",
        "usage": "自有摘要和结构假设；不复制论文全文或第三方代码。",
    },
    "model_names": ["ridge_mse", "patch_tst_mse", "autoformer_mse", "fedformer_mse", "mlp_mixer_mse"],
    "mlp_mixer_estimator": estimator.__class__.__name__,
    "mlp_mixer_has_torch_model": hasattr(estimator, "model_"),
    "mlp_mixer_lookback": estimator.lookback_,
    "mlp_mixer_feature_count": estimator.feature_count_,
    "mlp_mixer_token_hidden_size": estimator.token_hidden_size,
    "mlp_mixer_channel_hidden_size": estimator.channel_hidden_size,
    "constant_prediction": constant_prediction,
    "mlp_mixer_paper_fidelity": capability["paper_fidelity"],
    "mlp_mixer_is_proxy": capability["is_proxy"],
    "model_fidelity": model_fidelity_summary(("ridge_mse", "patch_tst_mse", "autoformer_mse", "fedformer_mse", "mlp_mixer_mse")),
    "top_model": comparison[0]["model"],
    "mlp_mixer_return_pct": float(mixer_row["return_pct"]),
    "mlp_mixer_relative_return_pct": float(mixer_row["relative_return_pct"]),
    "mlp_mixer_max_drawdown_pct": float(mixer_row["max_drawdown_pct"]),
    "mlp_mixer_sharpe": float(mixer_row["sharpe"]),
    "analysis_summary": outputs.summary.get("analysis_summary", {}),
    "artifact_links": {
        "matrix_card": "matrix_card/",
        "model_comparison": "matrix_card/model_comparison.csv",
        "model_run_ledger": "matrix_card/model_run_ledger.csv",
        "model_capabilities": "matrix_card/model_capabilities.csv",
        "analysis_manifest": "matrix_card/analysis_manifest.json",
        "artifact_manifest": "matrix_card/artifact_manifest.json",
    },
}
(OUTPUT_DIR / "notebook_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
summary


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'research.mlp_mixer_native_adapter_v1',
 'status': 'completed',
 'paper_reference': {'source_id': 'arxiv:2105.01601',
  'url': 'https://arxiv.org/abs/2105.01601',
  'usage': '自有摘要和结构假设；不复制论文全文或第三方代码。'},
 'model_names': ['ridge_mse',
  'patch_tst_mse',
  'autoformer_mse',
  'fedformer_mse',
  'mlp_mixer_mse'],
 'mlp_mixer_estimator': 'MLPMixerSequenceRegressor',
 'mlp_mixer_has_torch_model': True,
 'mlp_mixer_lookback': 5,
 'mlp_mixer_feature_count': 5,
 'mlp_mixer_token_hidden_size': 8,
 'mlp_mixer_channel_hidden_size': 16,
 'constant_prediction': None,
 'mlp_mixer_paper_fidelity': 'native_sequence',
 'mlp_mixer_is_proxy': False,
 'model_fidelity': {'schema_version': 'model.fidelity.v1',
  'model_count': 5,
  'fidelity_counts': {'native_library': 1, 'native_sequence': 4},
  'proxy_models': [],
  'native_models': ['autoformer_mse',
   'fedformer_mse',
   'mlp_mixer_mse',
   'patch_tst_mse',
   'ridge_mse'],
  'promotion_blocker_count': 4},
 'top_model': 'mlp_mixer_mse

## 中文结论

- 做了什么：在 `src/model.sequence` 增加 `MLPMixerSequenceRegressor`，把 `mlp_mixer_mse` 从 `proxy_feature_adapter` 升级为 `native_sequence`。实现保留 MLP-Mixer 的核心结构假设：时间步作为 token、因子列作为 channel，分别经过 token-mixing MLP 和 channel-mixing MLP，再用 head 输出收益预测。
- 核心指标：本 notebook 训练并回测 `ridge_mse`、`patch_tst_mse`、`autoformer_mse`、`fedformer_mse`、`mlp_mixer_mse`，输出 `model_comparison.csv`、`daily_backtest.csv`、`model_run_ledger.csv`、`qlib_risk_metrics.csv`、`model_effect_deltas.csv` 和 manifest。具体收益、回撤、Sharpe 以 `notebook_summary.json` 与 `matrix_card/analysis_summary.json` 为准。
- regime 差异：本轮复用 `src/analysis` 的 window/regime、qlib risk、model effect delta；MLP-Mixer 是否有价值不能只看全区间收益，要重点看 stress/downtrend 下的相对收益、回撤、稳定性和换手。
- 是否晋升：否。`native_sequence` 只表示实现形态已从 proxy 推进为稳定训练接口；production 晋升仍需要更长历史窗口、更多市场 regime、成本压力和 paper/sim replay 复验。
- 风险：当前是紧凑实现，不等价于完整论文训练规模；SOXL 单标的样本短，token/channel mixing 容易过拟合近期结构；外部论文仅作为结构参考，未复制第三方代码或全文。
- 下一步：刷新全模型 ledger，使 native/proxy 计数同步；继续验证 `sfm_mse`、`hawkes_gru_mse` 和 graph proxy replacement。